In [ ]:
import pandas as pd 
import numpy as np
import tensorflow as tf 
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers,models
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
import keras_tuner as kt

In [3]:
df_train = pd.read_csv(r'../data/selected_col/model_train.csv')
df_test = pd.read_csv(r'../data/selected_col/model_test.csv')
df_val = pd.read_csv(r'../data/selected_col/model_val.csv')

In [4]:
img_size = (224,224)
batch_size = 32
batch_size_64 = 64

In [5]:
data_augmentation_2 = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.2),
])

In [6]:
def preprocess_image2(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, img_size)
    return image, label

In [7]:
def preprocess_train2(image_path, label):
    image, label = preprocess_image2(image_path, label)
    image = data_augmentation_2(image)
    return image, label

In [8]:
def preprocess_test2(image_path, label):
    image, label = preprocess_image2(image_path, label)
    return image, label

In [9]:
train_dataset2 = tf.data.Dataset.from_tensor_slices(
    (
        df_train["path"].values,
        df_train["dx_encode"].values
    )
)

train_dataset2 = (
    train_dataset2
    .map(preprocess_train2, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [10]:
test_dataset2 = tf.data.Dataset.from_tensor_slices(
    (
        df_test["path"].values,
        df_test["dx_encode"].values
    )
)

test_dataset2 = (
    test_dataset2
    .map(preprocess_test2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [11]:
val_dataset2 = tf.data.Dataset.from_tensor_slices(
    (
        df_val["path"].values,
        df_val["dx_encode"].values
    )
)

val_dataset2 = (
    val_dataset2
    .map(preprocess_test2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)


In [12]:
#dataset 2 64

In [13]:
train_dataset2_64 = tf.data.Dataset.from_tensor_slices(
    (
        df_train["path"].values,
        df_train["dx_encode"].values
    )
)

train_dataset2_64 = (
    train_dataset2_64
    .map(preprocess_train2, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(batch_size_64)
    .prefetch(tf.data.AUTOTUNE)
)

In [14]:
test_dataset2_64 = tf.data.Dataset.from_tensor_slices(
    (
        df_test["path"].values,
        df_test["dx_encode"].values
    )
)

test_dataset2_64 = (
    test_dataset2_64
    .map(preprocess_test2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size_64)
    .prefetch(tf.data.AUTOTUNE)
)

In [15]:
val_dataset2_64 = tf.data.Dataset.from_tensor_slices(
    (
        df_val["path"].values,
        df_val["dx_encode"].values
    )
)

val_dataset2_64 = (
    val_dataset2_64
    .map(preprocess_test2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)


In [16]:
y_train = df_train["dx_encode"].values


class_weights = compute_class_weight(class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train)

# Convert to dictionary
class_weight_dict = dict(enumerate(class_weights))

class_weight_dict

{0: np.float64(4.372426699937617),
 1: np.float64(2.7813492063492062),
 2: np.float64(1.3020620471855842),
 3: np.float64(12.51607142857143),
 4: np.float64(1.2853475151292866),
 5: np.float64(0.2133572798392743),
 6: np.float64(10.113997113997113)}

In [17]:
'''CNN BASELINE WITH EARLY STOP'''

early_stop = EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True,verbose=1)

In [18]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

In [ ]:
'''early stop'''

In [19]:
densebase_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

densebase_model.trainable = False

dnmodel = tf.keras.Sequential([
    densebase_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7, activation="softmax")
])

dnmodel.summary()

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,301,703 (27.85 MB)

 Trainable params: 264,199 (1.01 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [20]:
dnmodel.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [22]:
history = dnmodel.fit(
    train_dataset2,
    validation_data=val_dataset2,
    epochs=5,
    class_weight=class_weight_dict,
    callbacks = [early_stop]
)

Epoch 1/5


c:\Users\nares\.virtualenvs\week_7-FuDx5imr\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 362s 2s/step - accuracy: 0.3210 - loss: 3.0559 - val_accuracy: 0.3593 - val_loss: 1.7934
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 348s 2s/step - accuracy: 0.3859 - loss: 1.7307 - val_accuracy: 0.2522 - val_loss: 1.7542
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 313s 1s/step - accuracy: 0.4169 - loss: 1.6385 - val_accuracy: 0.2269 - val_loss: 2.1904
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 362s 2s/step - accuracy: 0.4103 - loss: 1.6316 - val_accuracy: 0.3812 - val_loss: 1.6429
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 308s 1s/step - accuracy: 0.4369 - loss: 1.4996 - val_accuracy: 0.3147 - val_loss: 1.7398
Restoring model weights from the end of the best epoch: 4.


In [34]:
dnmodel.evaluate(test_dataset2)

47/47 ━━━━━━━━━━━━━━━━━━━━ 53s 1s/step - accuracy: 0.3925 - loss: 1.6467


[1.6466833353042603, 0.39254823327064514]

In [35]:
dnmodel.save(r'../models/dense_early.keras')

In [ ]:
#learning rate

In [23]:
densebase_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

densebase_model.trainable = False

dnmodel_lr = tf.keras.Sequential([
    densebase_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7, activation="softmax")
])

dnmodel_lr.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,301,703 (27.85 MB)

 Trainable params: 264,199 (1.01 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [24]:
dnmodel_lr.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [25]:
history = dnmodel_lr.fit(train_dataset2,validation_data=val_dataset2,epochs=5,class_weight=class_weight_dict,callbacks = [early_stop,lr_scheduler])

Epoch 1/5


c:\Users\nares\.virtualenvs\week_7-FuDx5imr\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 308s 1s/step - accuracy: 0.3641 - loss: 3.1401 - val_accuracy: 0.4152 - val_loss: 1.6137 - learning_rate: 0.0010
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 403s 2s/step - accuracy: 0.4239 - loss: 1.9776 - val_accuracy: 0.2309 - val_loss: 2.3464 - learning_rate: 0.0010
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 315s 1s/step - accuracy: 0.4267 - loss: 1.6261 - val_accuracy: 0.5030 - val_loss: 1.2333 - learning_rate: 0.0010
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 316s 1s/step - accuracy: 0.4470 - loss: 1.5532 - val_accuracy: 0.4478 - val_loss: 1.3157 - learning_rate: 0.0010
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.4648 - loss: 1.4637
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
220/220 ━━━━━━━━━━━━━━━━━━━━ 951s 4s/step - accuracy: 0.4648 - loss: 1.4637 - val_accuracy: 0.1557 - val_loss: 2.0126 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 3.


In [26]:
dnmodel_lr.evaluate(test_dataset2)

47/47 ━━━━━━━━━━━━━━━━━━━━ 60s 1s/step - accuracy: 0.4983 - loss: 1.2669


[1.2668874263763428, 0.49833667278289795]

In [36]:
dnmodel_lr.save(r'../models/dense_learning.keras')

In [ ]:
#optimer 
#adam

In [28]:
densebase_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

densebase_model.trainable = False

dnmodel_adam = tf.keras.Sequential([
    densebase_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7, activation="softmax")
])

dnmodel_adam.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,301,703 (27.85 MB)

 Trainable params: 264,199 (1.01 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [29]:
dnmodel_adam.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
history = dnmodel_adam.fit(train_dataset2,validation_data=val_dataset2,epochs=5,class_weight=class_weight_dict,callbacks = [early_stop])

Epoch 1/5


c:\Users\nares\.virtualenvs\week_7-FuDx5imr\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 383s 2s/step - accuracy: 0.3233 - loss: 2.9446 - val_accuracy: 0.0852 - val_loss: 3.0791
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 304s 1s/step - accuracy: 0.4109 - loss: 1.8287 - val_accuracy: 0.2049 - val_loss: 2.0527
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 301s 1s/step - accuracy: 0.4273 - loss: 1.6726 - val_accuracy: 0.3194 - val_loss: 1.6099
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 306s 1s/step - accuracy: 0.4514 - loss: 1.5320 - val_accuracy: 0.4584 - val_loss: 1.4799
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 308s 1s/step - accuracy: 0.4476 - loss: 1.4967 - val_accuracy: 0.4092 - val_loss: 1.4606
Restoring model weights from the end of the best epoch: 5.


In [30]:
dnmodel_adam.evaluate(test_dataset2)

47/47 ━━━━━━━━━━━━━━━━━━━━ 50s 1s/step - accuracy: 0.4185 - loss: 1.4439


[1.4439339637756348, 0.41849634051322937]

In [38]:
dnmodel_adam.save(r'../models/dense_adam.keras')

In [ ]:
#sgd

In [31]:
densebase_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

densebase_model.trainable = False

dnmodel_sgd = tf.keras.Sequential([
    densebase_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7, activation="softmax")
])

dnmodel_sgd.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,301,703 (27.85 MB)

 Trainable params: 264,199 (1.01 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [32]:
dnmodel_sgd.compile(optimizer="sgd",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
history = dnmodel_sgd.fit(train_dataset2,validation_data=val_dataset2,epochs=5,class_weight=class_weight_dict,callbacks = [early_stop])

Epoch 1/5


c:\Users\nares\.virtualenvs\week_7-FuDx5imr\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 291s 1s/step - accuracy: 0.2632 - loss: 3.4468 - val_accuracy: 0.4391 - val_loss: 1.7360
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 284s 1s/step - accuracy: 0.2200 - loss: 1.9091 - val_accuracy: 0.6174 - val_loss: 1.9145
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 286s 1s/step - accuracy: 0.3558 - loss: 1.9097 - val_accuracy: 0.3273 - val_loss: 2.2564
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 289s 1s/step - accuracy: 0.3427 - loss: 1.9056 - val_accuracy: 0.0366 - val_loss: 2.0514
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.


In [33]:
dnmodel_sgd.evaluate(test_dataset2)

47/47 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - accuracy: 0.4444 - loss: 1.7481


[1.7481248378753662, 0.4444444477558136]

In [40]:
dnmodel_sgd.save(r'../models/dnmodel_sgd.keras')

history_df = pd.DataFrame(history.history)


history_df.to_csv(r'../log/dense__sgd.csv', index=False)

In [ ]:
#rms

In [41]:
densebase_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

densebase_model.trainable = False

dnmodel_rms = tf.keras.Sequential([
    densebase_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7, activation="softmax")
])

dnmodel_rms.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,301,703 (27.85 MB)

 Trainable params: 264,199 (1.01 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [42]:
dnmodel_rms.compile(optimizer=tf.keras.optimizers.RMSprop(),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
history = dnmodel_rms.fit(train_dataset2,validation_data=val_dataset2,epochs=5,class_weight=class_weight_dict,callbacks = [early_stop])

Epoch 1/5


c:\Users\nares\.virtualenvs\week_7-FuDx5imr\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 289s 1s/step - accuracy: 0.3242 - loss: 4.0275 - val_accuracy: 0.4225 - val_loss: 1.9561
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 280s 1s/step - accuracy: 0.3712 - loss: 2.2918 - val_accuracy: 0.6021 - val_loss: 1.1005
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 285s 1s/step - accuracy: 0.3979 - loss: 1.9995 - val_accuracy: 0.5808 - val_loss: 1.1238
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 287s 1s/step - accuracy: 0.4316 - loss: 1.8359 - val_accuracy: 0.3426 - val_loss: 3.3614
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 297s 1s/step - accuracy: 0.4342 - loss: 1.9067 - val_accuracy: 0.3906 - val_loss: 2.0400
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.


In [43]:
dnmodel_rms.evaluate(test_dataset2)

47/47 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - accuracy: 0.5915 - loss: 1.0935


[1.0934727191925049, 0.5914837121963501]

In [44]:
dnmodel_sgd.save(r'../models/dnmodel_rms.keras')

history_df = pd.DataFrame(history.history)


history_df.to_csv(r'../log/dense_rms.csv', index=False)

In [ ]:
#64

In [45]:
densebase_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

densebase_model.trainable = False

dnmodel_64 = tf.keras.Sequential([
    densebase_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dense(7, activation="softmax")
])

dnmodel_64.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_5      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,301,703 (27.85 MB)

 Trainable params: 264,199 (1.01 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [46]:
dnmodel_64.compile(optimizer=tf.keras.optimizers.RMSprop(),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
history = dnmodel_64.fit(train_dataset2_64,validation_data=val_dataset2_64,epochs=5,class_weight=class_weight_dict,callbacks = [early_stop])

Epoch 1/5


c:\Users\nares\.virtualenvs\week_7-FuDx5imr\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


110/110 ━━━━━━━━━━━━━━━━━━━━ 331s 3s/step - accuracy: 0.2925 - loss: 4.3888 - val_accuracy: 0.0206 - val_loss: 5.9458
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 312s 3s/step - accuracy: 0.3674 - loss: 2.1626 - val_accuracy: 0.1477 - val_loss: 2.6920
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 334s 3s/step - accuracy: 0.3882 - loss: 1.9214 - val_accuracy: 0.2269 - val_loss: 1.7492
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 321s 3s/step - accuracy: 0.4092 - loss: 1.7426 - val_accuracy: 0.3273 - val_loss: 1.7512
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 338s 3s/step - accuracy: 0.4326 - loss: 1.6465 - val_accuracy: 0.4943 - val_loss: 1.3024
Restoring model weights from the end of the best epoch: 5.


In [47]:
dnmodel_64.evaluate(test_dataset2_64)

24/24 ━━━━━━━━━━━━━━━━━━━━ 62s 3s/step - accuracy: 0.4997 - loss: 1.3228


[1.322758674621582, 0.49966734647750854]

In [48]:
dnmodel_64.save(r'../models/dense_64.keras')

In [ ]:
#hyper parameter

In [49]:
INPUT_SHAPE = (224,224,3)
NUM_CLASSES = 7

def build_densenet(hp):

    base_model = DenseNet121(
        weights="imagenet",
        include_top=False,
        input_shape=INPUT_SHAPE
    )

    base_model.trainable = False

    model = models.Sequential([

        base_model,

        layers.GlobalAveragePooling2D(),

        layers.Dense(
            hp.Choice(
                "dense_units",
                values=[128,256,512]
            ),
            activation="relu"
        ),

        layers.Dropout(
            hp.Choice(
                "dropout",
                values=[0.2,0.3,0.5]
            )
        ),

        layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        )

    ])


    model.compile(
        optimizer=hp.Choice(
        "optimizer",
        values=["adam","rmsprop","sgd"]
    ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [51]:
tuner = kt.RandomSearch(
    build_densenet,
    objective="val_accuracy",
    max_trials=5,
    overwrite=True,
    directory="hyperparameter_tuning",
    project_name="densenet121"
)

In [52]:
tuner.search(
    train_dataset2,
    validation_data=val_dataset2,
    epochs=5,
    class_weight=class_weight_dict
)

Trial 5 Complete [00h 27m 54s]
val_accuracy: 0.17298735678195953

Best val_accuracy So Far: 0.5981370806694031
Total elapsed time: 02h 56m 11s


In [53]:
best_hps = tuner.get_best_hyperparameters(1)[0]

print(best_hps.values)

{'dense_units': 512, 'dropout': 0.3, 'optimizer': 'rmsprop'}


In [54]:
best_model = tuner.get_best_models(1)[0]

c:\Users\nares\.virtualenvs\week_7-FuDx5imr\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(store)


In [55]:
history = best_model.fit(
    train_dataset2,
    validation_data=val_dataset2,
    epochs=5,
    class_weight=class_weight_dict,
    callbacks=[early_stop]
)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 317s 1s/step - accuracy: 0.4269 - loss: 1.9242 - val_accuracy: 0.6287 - val_loss: 1.0134
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 307s 1s/step - accuracy: 0.4431 - loss: 1.8783 - val_accuracy: 0.4757 - val_loss: 1.3330
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 312s 1s/step - accuracy: 0.4474 - loss: 1.7965 - val_accuracy: 0.5210 - val_loss: 1.2882
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 313s 1s/step - accuracy: 0.4286 - loss: 1.9350 - val_accuracy: 0.4025 - val_loss: 1.5716
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.


In [56]:
test_loss, test_accuracy = best_model.evaluate(test_dataset2)

print("Test Accuracy :", test_accuracy)
print("Test Loss :", test_loss)

47/47 ━━━━━━━━━━━━━━━━━━━━ 52s 1s/step - accuracy: 0.6121 - loss: 1.0192
Test Accuracy : 0.6121091246604919
Test Loss : 1.0191785097122192


In [57]:
best_model.save(r'../models/dense_hyper.keras')